# Lab 08: DDPM on MNIST

**Module 08** | Read `notes/08-ddpm-stable-diffusion.pdf` before running this notebook.

Train a small noise-predictor U-Net on MNIST and sample images by reversing the diffusion process.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Lab 08: DDPM on MNIST

A **DDPM** (Denoising Diffusion Probabilistic Model) trains a neural network to **predict the noise** added during forward diffusion. After training, **sampling** (the **reverse process**) starts from pure noise and repeatedly denoises to synthesize new digits.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **DDPM** | A diffusion model that learns to remove noise step by step to generate images. |
| **Noise prediction** | The network's job: given a noisy image at timestep t, output the epsilon noise that was mixed in. |
| **U-Net** | An hourglass-shaped CNN with skip connections. Common architecture for image-to-image tasks like denoising. |
| **Timestep embedding** | A vector telling the network which noise level (which t) it is looking at. |
| **MSE loss** | Mean Squared Error: average squared difference between predicted and true noise. |
| **q_sample** | Helper that applies the forward diffusion formula to a clean batch at random timesteps. |
| **Reverse process (sampling)** | Generation: start at x_T (pure noise), predict noise, estimate cleaner x, repeat down to t=0. |
| **Posterior mean** | The statistically best guess for x_{t-1} given x_t and the predicted clean image. |
| **GroupNorm** | Normalizes channels per image (not per batch). Useful when batch size is small. |
| **Subset training** | Training on 8,000 images instead of 60,000 to finish faster in a classroom setting. |

Refer back to this table whenever a term appears in code or output.


### Concept: What the model learns

**Forward diffusion (review):** `x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon`

**Why we add noise:** It creates a smooth path from real images to random noise. The network practices denoising at every point along that path.

**What the U-Net predicts:** Not the digit directly. It predicts **epsilon**, the same random noise vector that was added. If you know x_t and epsilon, you can algebraically recover an estimate of the clean x_0.

**Noise prediction loss:** Sample random t, corrupt a real image, ask the network to output epsilon_hat, minimize `MSE(epsilon_hat, epsilon)`.

**Sampling (reverse process):** Begin with `x_T ~ N(0, I)`. For t from T-1 down to 0: (1) predict noise, (2) estimate x_0, (3) compute the mean of x_{t-1}, (4) add a small random step (except at t=0). After T steps you get a generated digit.


### Step 1: Schedule, data subset, and diffusion helpers

**What this does:** Builds a 200-step beta schedule, loads an 8,000-image MNIST subset, and prints training configuration.

**Why we do this:** Fewer timesteps (200 vs 500) and a subset keep runtime manageable while preserving the full DDPM training pattern.


**What to look for in the output**

Subset 8,000 images, batch size 128, T=200, 3 epochs. alpha_bar_T should be a tiny positive number. Batches per epoch should be 62 (8000/128).


In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
T = 200  # fewer steps than Lab 07 for faster training/sampling
BATCH_SIZE = 128
EPOCHS = 3
LR = 2e-4
SUBSET_SIZE = 8000

# --- Diffusion schedule tensors (same logic as Lab 07) ---
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
full_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
train_set = Subset(full_set, list(range(SUBSET_SIZE)))  # first 8k images only
loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("DDPM training setup")
print(f"  Full MNIST train:  {len(full_set):,} images")
print(f"  Subset used:       {len(train_set):,} images (indices 0..{SUBSET_SIZE - 1})")
print(f"  Batch size:        {BATCH_SIZE}")
print(f"  Batches per epoch: {len(loader)}")
print(f"  Timesteps T:       {T}")
print(f"  Epochs:            {EPOCHS}")
print(f"  beta_1 / beta_T:   {betas[0]:.6f} / {betas[-1]:.4f}")
print(f"  alpha_bar_T:       {alpha_bars[-1]:.6f}")


### Step 2: Define the noise-predictor U-Net

**What this does:** Builds a compact U-Net with sinusoidal timestep embeddings, residual blocks, and a `q_sample` helper for forward noising.

**Why we do this:** The U-Net sees both the noisy image and the timestep t. Skip connections help preserve thin stroke detail when denoising MNIST digits.


**What to look for in the output**

Prints noise predictor parameter count (on the order of 200k-400k). Classes `SimpleUNet` and function `q_sample` should be defined.


In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Maps integer timestep t to a dense vector (like a clock encoding)."""

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / half
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2:
            emb = F.pad(emb, (0, 1))
        return self.proj(emb)


class ResBlock(nn.Module):
    """Convolution block that adds timestep information, then residual-adds input."""

    def __init__(self, channels: int, time_dim: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.time = nn.Linear(time_dim, channels)  # broadcast time embedding into channels
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.conv2(F.silu(self.norm2(h)))
        return x + h  # residual connection stabilizes deep nets


class SimpleUNet(nn.Module):
    """Predicts noise epsilon given noisy image x_t and timestep t."""

    def __init__(self, time_dim: int = 64):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(time_dim)
        self.in_conv = nn.Conv2d(1, 32, 3, padding=1)
        self.down1 = ResBlock(32, time_dim)
        self.down2 = ResBlock(32, time_dim)
        self.pool = nn.MaxPool2d(2)
        self.mid = ResBlock(64, time_dim)
        self.up_conv = nn.Conv2d(64, 32, 3, padding=1)
        self.up1 = ResBlock(32, time_dim)
        self.up2 = ResBlock(32, time_dim)
        self.out_conv = nn.Conv2d(32, 1, 1)  # output same shape as input noise

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_emb(t)
        h0 = self.in_conv(x)
        h1 = self.down1(h0, t_emb)
        h2 = self.down2(h1, t_emb)
        h = self.pool(h2)
        h = torch.cat([h, self.pool(h1)], dim=1)  # skip connection from encoder
        h = self.mid(h, t_emb)
        h = F.interpolate(h, scale_factor=2, mode="nearest")  # upsample decoder
        h = self.up_conv(h) + h2  # another skip
        h = self.up1(h, t_emb)
        h = self.up2(h, t_emb)
        return self.out_conv(h)  # predicted noise map


def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    """Forward diffusion: mix clean x0 and noise at timesteps t (per image in batch)."""
    sa = sqrt_alpha_bars.to(x0.device)[t].view(-1, 1, 1, 1)
    so = sqrt_one_minus_alpha_bars.to(x0.device)[t].view(-1, 1, 1, 1)
    return sa * x0 + so * noise


model = SimpleUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print(f"Noise predictor parameters: {sum(p.numel() for p in model.parameters()):,}")


### Step 3: Walk through one training batch

**What this does:** Takes one batch, picks random timesteps, adds noise, runs the U-Net once, and prints MSE between predicted and true noise.

**Why we do this:** Mirrors the inner training step so you see exactly what the loss measures before running the full loop.


**What to look for in the output**

Batch shape (128, 1, 28, 28). Sampled t is an integer 0-199. MSE values are small but not zero on an untrained model. Printed goal reminds you training should drive batch MSE down.


In [ ]:
torch.manual_seed(0)
demo_images, _ = next(iter(loader))
demo_images = demo_images.to(device)
demo_noise = torch.randn_like(demo_images)  # ground-truth epsilon
demo_t = torch.randint(0, T, (demo_images.size(0),), device=device)  # random t per image
demo_x_t = q_sample(demo_images, demo_t, demo_noise)  # corrupt the batch

model.eval()
with torch.no_grad():
    demo_pred = model(demo_x_t, demo_t)  # network guesses the noise

demo_mse = F.mse_loss(demo_pred, demo_noise)
idx = 0
single_mse = F.mse_loss(demo_pred[idx], demo_noise[idx])

print("Single-batch training demo")
print(f"  Batch shape:        {tuple(demo_images.shape)}")
print(f"  Example index:      {idx}")
print(f"  Sampled t:          {demo_t[idx].item()}")
print(f"  alpha_bar at t:     {alpha_bars.to(device)[demo_t[idx]].item():.6f}")
print(f"  MSE (one example):  {single_mse.item():.6f}")
print(f"  MSE (full batch):   {demo_mse.item():.6f}")
print("  Goal during training: drive batch MSE down across random timesteps.")


### Step 4: Full training loop

**What this does:** Trains for 3 epochs. Each step: random t, forward noise, predict noise, backprop MSE loss.

**Why we do this:** This is standard DDPM training. Falling average MSE means the U-Net is learning the noise prediction task.


**What to look for in the output**

Three epoch lines with decreasing (or mostly decreasing) average noise MSE. A loss plot with 3 points should appear. Values often drop from roughly 0.1-0.3 toward 0.05-0.15 depending on hardware.


In [ ]:
loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for images, _ in loader:
        images = images.to(device)
        noise = torch.randn_like(images)  # true epsilon to predict
        t = torch.randint(0, T, (images.size(0),), device=device)
        x_t = q_sample(images, t, noise)  # create noisy training inputs

        optimizer.zero_grad()
        pred_noise = model(x_t, t)  # U-Net output = epsilon_hat
        loss = F.mse_loss(pred_noise, noise)  # noise prediction objective
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(loader)
    loss_history.append(avg)
    print(f"Epoch {epoch}/{EPOCHS}, average noise MSE: {avg:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("DDPM noise-prediction loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Concept: Reverse diffusion sampling

**Sampling** means generating new images. We reverse the forward chain:

1. Start with `x_T` drawn from pure Gaussian noise.
2. For `t = T-1, T-2, ... , 0`:
   - Feed `(x_t, t)` to the U-Net to get `pred_noise`.
   - Estimate clean `x_0` from `x_t` and `pred_noise` using the forward formula solved for x_0.
   - Compute the mean of the posterior `x_{t-1}`.
   - Add a small Gaussian perturbation (except at the last step).
3. At `t=0`, output the final `x_0` as the generated digit.

This loop is slower than a single GAN forward pass but tends to produce stable, diverse samples.


### Step 5: Reverse diffusion sampling

**What this does:** Implements `sample_ddpm`, generates 16 images, prints statistics, and displays a 4x4 grid.

**Why we do this:** This is the payoff: images from noise. Quality is rough after only 3 epochs but you should see blobby digit shapes.


**What to look for in the output**

16 images shaped (16, 1, 28, 28). Mean near 0, std modest. Grid shows gray digit-like forms. More training would sharpen strokes.


In [ ]:
@torch.no_grad()
def sample_ddpm(n: int = 16) -> torch.Tensor:
    """Generate n images by reversing the diffusion process from pure noise."""
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)  # start at x_T
    for step in reversed(range(T)):
        t = torch.full((n,), step, device=device, dtype=torch.long)
        pred_noise = model(x, t)  # predict epsilon at this noise level
        ab = alpha_bars[step].to(device)
        ab_prev = alpha_bars[step - 1].to(device) if step > 0 else torch.tensor(1.0, device=device)
        beta = betas[step].to(device)
        alpha = alphas[step].to(device)

        # Recover x_0 estimate from x_t and predicted noise (rearrange forward formula)
        x0_pred = (x - torch.sqrt(1.0 - ab) * pred_noise) / torch.sqrt(ab)
        x0_pred = x0_pred.clamp(-1.0, 1.0)  # keep pixels in valid range

        if step > 0:
            # Posterior mean for x_{t-1} (DDPM sampling formula)
            coef1 = torch.sqrt(ab_prev) * beta / (1.0 - ab)
            coef2 = torch.sqrt(alpha) * (1.0 - ab_prev) / (1.0 - ab)
            mean = coef1 * x0_pred + coef2 * x
            noise = torch.randn_like(x)
            x = mean + torch.sqrt(beta) * noise  # small stochastic step
        else:
            x = x0_pred  # final step: no extra noise
    return x


samples = sample_ddpm(16).cpu()
print("Generated sample statistics (16 images)")
print(f"  Shape:      {tuple(samples.shape)}")
print(f"  Mean:       {samples.mean():.4f}")
print(f"  Std:        {samples.std():.4f}")
print(f"  Min / max:  {samples.min():.3f} / {samples.max():.3f}")
print("  After more epochs, means stay near 0 but digit structure becomes sharper.")

grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(5, 5))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("DDPM samples (16 images)")
plt.axis("off")
plt.tight_layout()
plt.show()
